# LeetCode 611 — Valid Triangle Number bằng Python

Link đề: https://leetcode.com/problems/valid-triangle-number/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Two Pointers, Binary Search, Sorting

Skeleton LeetCode:

```python
class Solution:
    def triangleNumber(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **bộ ba chỉ số** `i < j < k` sao cho ba số `nums[i], nums[j], nums[k]` làm được ba cạnh của một tam giác (có tính trùng giá trị nhưng khác chỉ số là các bộ khác nhau).
2. Nắm các công cụ Python: **bất đẳng thức tam giác**, `sorted`/`list.sort`, `bisect_left` (kèm tham số `lo`), kỹ thuật **hai con trỏ**, `itertools.combinations`, và công thức đếm tổ hợp `C(n, 3)`.
3. Đi từ **vét cạn ba vòng lặp** → **sắp xếp + tìm nhị phân vị trí cạnh thứ ba** → **sắp xếp + hai con trỏ** → **bảng đếm trước (prefix count) thay cho tìm nhị phân**.
4. Hiểu bước đột phá: sau khi **sắp xếp** `a ≤ b ≤ c`, chỉ còn **một** điều kiện cần kiểm tra là `a + b > c`; và với `c` cố định, các cặp `(a, b)` thoả điều kiện có tính **đơn điệu** nên đếm được cả loạt bằng hai con trỏ.
5. Tránh các lỗi kinh điển: dùng `>=` (tam giác suy biến), không sắp xếp, cộng sai số lượng khi hai con trỏ, loại trùng bằng `set`, và đếm mỗi bộ ba nhiều lần.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trả về **số bộ ba** chọn từ mảng mà nếu lấy ba số đó làm **độ dài ba cạnh** thì tạo thành một **tam giác** (hợp lệ, không suy biến).

**Ví dụ 1:** `nums = [2,2,3,4]` ⇒ `3`. Các bộ hợp lệ: `(2,3,4)` dùng số `2` đầu tiên; `(2,3,4)` dùng số `2` thứ hai; `(2,2,3)`. (Hai bộ `(2,3,4)` được tính là **khác nhau** vì lấy từ hai chỉ số khác nhau.)

**Ví dụ 2:** `nums = [4,2,3,4]` ⇒ `4`.

**Ràng buộc:**

- `1 <= nums.length <= 1000`
- `0 <= nums[i] <= 1000`

Chú ý: có thể có số `0` (cạnh dài 0 không bao giờ tạo tam giác) và có thể có nhiều số bằng nhau.

Cell dưới nạp thư viện và hai ví dụ (chưa giải).

In [1]:
import random
import time
from bisect import bisect_left
from itertools import combinations
from math import comb
from typing import List

examples = [([2, 2, 3, 4], 3), ([4, 2, 3, 4], 4)]
for nums, expected in examples:
    print(nums, "-> mong đợi", expected, "| số bộ ba chỉ số:", comb(len(nums), 3))

[2, 2, 3, 4] -> mong đợi 3 | số bộ ba chỉ số: 4
[4, 2, 3, 4] -> mong đợi 4 | số bộ ba chỉ số: 4


## 3. Kiến thức nền

### 3.1. Bất đẳng thức tam giác

Ba đoạn dài `a, b, c` dựng được tam giác **khi và chỉ khi** tổng hai cạnh bất kỳ **lớn hơn hẳn** cạnh còn lại:

- `a + b > c`, `a + c > b`, `b + c > a`.

Nếu **bằng nhau** (ví dụ `1 + 2 = 3`) thì ba đoạn "dẹt" lại thành một đoạn thẳng — **tam giác suy biến**, không tính.

In [2]:
def is_triangle(a, b, c):
    return a + b > c and a + c > b and b + c > a


print(is_triangle(3, 4, 5))     # True
print(is_triangle(1, 2, 3))     # False (1 + 2 == 3: suy biến)
print(is_triangle(2, 2, 3))     # True
print(is_triangle(0, 5, 5))     # False (cạnh 0)

True
False
True
False


### 3.2. Sắp xếp rút gọn ba điều kiện thành một

Nếu ba số đã sắp xếp `a ≤ b ≤ c` thì:

- `a + c ≥ b` (vì `c ≥ b` và `a ≥ 0`) — hơn nữa `a + c > b` khi `a > 0`; còn khi `a = 0` thì `a + b = b ≤ c` nên điều kiện chính `a + b > c` đã sai.
- `b + c ≥ a` tương tự.

Vậy **chỉ cần kiểm tra `a + b > c`** (cạnh nhỏ nhất + cạnh vừa > cạnh lớn nhất). Nếu `a + b > c` thì `c < a + b ≤ b + b`, suy ra `a > 0` (vì `c ≥ b`), nên hai điều kiện còn lại tự đúng. Cell dưới kiểm chứng bằng máy trên mọi bộ trong `[0, 12]`.

In [3]:
bad = 0
for a in range(13):
    for b in range(a, 13):
        for c in range(b, 13):
            if is_triangle(a, b, c) != (a + b > c):
                bad += 1
print("số bộ (a<=b<=c) mà hai cách kiểm tra khác nhau:", bad)
assert bad == 0

số bộ (a<=b<=c) mà hai cách kiểm tra khác nhau: 0


### 3.3. Sắp xếp và đếm theo **chỉ số**

`nums.sort()` sắp xếp tại chỗ, `sorted(nums)` trả list mới. Đề đếm **bộ ba chỉ số**: hai số bằng nhau ở hai vị trí khác nhau vẫn là hai "cạnh" khác nhau. Sắp xếp **không làm mất** thông tin đó (vẫn còn hai phần tử), nên ta đếm bình thường trên mảng đã sắp xếp.

In [4]:
nums = [4, 2, 3, 4]
s = sorted(nums)
print(s, "| nums gốc không đổi:", nums)
nums.sort()
print(nums)

[2, 3, 4, 4] | nums gốc không đổi: [4, 2, 3, 4]
[2, 3, 4, 4]


### 3.4. `bisect_left(a, x, lo)` — tìm nhị phân vị trí đầu tiên `≥ x`

Với mảng `a` đã sắp xếp, `bisect_left(a, x)` trả **chỉ số đầu tiên** mà `a[idx] >= x` (hoặc `len(a)` nếu mọi phần tử `< x`). Tham số `lo` giới hạn tìm từ chỉ số `lo` trở đi: `bisect_left(a, x, lo)`. Độ phức tạp `O(log n)`.

In [5]:
a = [2, 2, 3, 4, 4, 7]
print(bisect_left(a, 4))        # 3: số 4 đầu tiên ở chỉ số 3
print(bisect_left(a, 5))        # 5: vị trí chèn 5 (đứng trước 7)
print(bisect_left(a, 10))       # 6: lớn hơn hết
print(bisect_left(a, 2, 1))     # 1: bắt đầu tìm từ chỉ số 1

3
5
6
1


### 3.5. Kỹ thuật hai con trỏ trên mảng đã sắp xếp

Đặt con trỏ `i` ở đầu, `j` ở cuối. Dựa vào việc so sánh `a[i] + a[j]` với một ngưỡng mà ta **chắc chắn** nên di chuyển con trỏ nào, nhờ tính đơn điệu của mảng sắp xếp. Mỗi bước loại bỏ ít nhất một ứng viên nên tổng `O(n)`. (Mục 6 áp dụng cụ thể.)

### 3.6. Số bộ ba chỉ số: `C(n, 3)`

Số cách chọn 3 chỉ số khác nhau không để ý thứ tự là `C(n, 3) = n(n-1)(n-2)/6`. Với `n = 1000` là `166 167 000` — con số cho thấy ba vòng lặp lồng nhau (`n³/6 ≈ 1.7·10^8` bước) chậm trong Python.

In [6]:
print(comb(4, 3), comb(1000, 3))
print(len(list(combinations(range(5), 3))), comb(5, 3))     # 10 10
print(list(combinations([2, 2, 3, 4], 3)))                  # theo chỉ số: hai bộ (2,3,4) khác nhau

4 166167000
10 10
[(2, 2, 3), (2, 2, 4), (2, 3, 4), (2, 3, 4)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Ví dụ 1:** `[2, 2, 3, 4]` có `C(4,3) = 4` bộ ba chỉ số. Xét từng bộ:

| bộ | giá trị | `a+b>c`? | tam giác? |
|---|---|---|---|
| (0,1,2) | 2,2,3 | 4 > 3 | ✓ |
| (0,1,3) | 2,2,4 | 4 > 4 sai | ✗ (suy biến) |
| (0,2,3) | 2,3,4 | 5 > 4 | ✓ |
| (1,2,3) | 2,3,4 | 5 > 4 | ✓ |

Có 3 bộ hợp lệ ⇒ đáp án `3`.

**Ví dụ 2:** `[4,2,3,4]` sắp xếp `[2,3,4,4]`, `C(4,3) = 4` bộ: `(2,3,4)` ✓, `(2,3,4)` ✓ (với số `4` kia), `(2,4,4)` ✓ (6 > 4), `(3,4,4)` ✓ ⇒ 4.

**Nhận xét quan trọng khi cạnh lớn nhất `c` cố định:** các cặp `(a, b)` nhỏ hơn thoả `a + b > c`. Nếu **nâng** `a` lên (hoặc nâng `b`), tổng chỉ tăng ⇒ vẫn thoả. Tính đơn điệu này cho phép đếm hàng loạt thay vì từng cặp.

## 5. Cách 1 — Brute Force / baseline: ba vòng lặp, kiểm tra cả ba bất đẳng thức

**Ý tưởng:** duyệt mọi `i < j < k`, lấy ba giá trị, kiểm tra đủ ba điều kiện `a+b>c`, `a+c>b`, `b+c>a`, đếm. Không cần sắp xếp. Cách này bám sát định nghĩa nên làm **oracle** để đối chiếu các cách sau.

In [7]:
def tri_bruteforce(nums):
    """Cách 1: ba vòng lặp, kiểm tra cả ba bất đẳng thức (không cần sort)."""
    n = len(nums)
    cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                a, b, c = nums[i], nums[j], nums[k]
                if a + b > c and a + c > b and b + c > a:
                    cnt += 1
    return cnt


for nums, expected in examples:
    print(nums, "->", tri_bruteforce(nums), "(mong đợi", expected, ")")
    assert tri_bruteforce(nums) == expected
print(tri_bruteforce([1, 2, 3]), tri_bruteforce([0, 0, 0]), tri_bruteforce([1, 1, 1]))   # 0 0 1

[2, 2, 3, 4] -> 3 (mong đợi 3 )
[4, 2, 3, 4] -> 4 (mong đợi 4 )
0 0 1


**Độ phức tạp:** thời gian **`O(n³)`** (chính xác `C(n,3)` lần kiểm tra), bộ nhớ `O(1)`.

**Nút thắt:** `n = 1000` ⇒ `1.66·10^8` bước, quá chậm trong Python (hàng chục giây). Hai chỗ lãng phí: (1) kiểm tra **ba** điều kiện trong khi chỉ cần một (mục 3.2 sau khi sắp xếp), (2) duyệt **từng** `k` trong khi các `k` hợp lệ liền nhau thành một đoạn.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Sắp xếp** `nums` tăng dần. Với `i < j < k` thì `nums[i] ≤ nums[j] ≤ nums[k]`, chỉ cần kiểm `nums[i] + nums[j] > nums[k]`.
2. **Cố định hai cạnh nhỏ** `i < j`. Các `k > j` hợp lệ là những `k` có `nums[k] < nums[i] + nums[j]`. Vì mảng tăng dần nên các `k` đó tạo thành **một đoạn liền** `[j+1, left-1]`, với `left` là chỉ số đầu tiên có `nums[left] >= nums[i] + nums[j]`. Số `k` hợp lệ = `left - j - 1`. ⇒ **tìm nhị phân** `left` (Cách 2): `O(n² log n)`.
3. **Hoặc cố định cạnh lớn nhất** `k` và đếm các cặp `(i, j)` với `i < j < k`, `nums[i] + nums[j] > nums[k]` bằng **hai con trỏ** `i = 0`, `j = k - 1`:
   - nếu `nums[i] + nums[j] > nums[k]` thì **mọi** `i' ∈ [i, j-1]` đều hợp lệ với `j` (vì `nums[i'] ≥ nums[i]`) ⇒ cộng `j - i` cặp, rồi `j -= 1`;
   - ngược lại `i += 1` (cạnh nhỏ quá, thử cạnh lớn hơn).
   Mỗi bước dịch một con trỏ ⇒ `O(k)` cho mỗi `k` ⇒ tổng **`O(n²)`** (Cách 3).
4. **Biến thể khai thác `nums[i] ≤ 1000`:** thay tìm nhị phân bằng bảng "có bao nhiêu phần tử `< s`" dựng trước (Cách 4): tra `O(1)`, tổng cũng `O(n²)`.

Ta sẽ lần lượt cài đặt từng cách.

## 7. Các cách tiếp theo

Tổng cộng 4 cách:

- Cách 1: ba vòng lặp (mục 5).
- Cách 2: sắp xếp + tìm nhị phân (`bisect_left`) — **lời giải chính thức của doocs**.
- Cách 3: sắp xếp + hai con trỏ (**tối ưu về độ phức tạp, dùng khi nộp**).
- Cách 4: sắp xếp + bảng đếm trước (prefix count) thay cho tìm nhị phân.

---

### Cách 2 — Sắp xếp + tìm nhị phân vị trí cạnh thứ ba

Giải thích từng đoạn:

1. `nums.sort()`.
2. Với `i` từ `0` đến `n-3`, `j` từ `i+1` đến `n-2`: ngưỡng `s = nums[i] + nums[j]`.
3. `left = bisect_left(nums, s, j + 1)`: chỉ số đầu tiên (từ `j+1`) có giá trị `>= s`.
4. Các `k ∈ [j+1, left-1]` đều hợp lệ ⇒ `ans += left - j - 1`.

(Nếu `s` rất nhỏ thì `left = j + 1`, cộng `0`.)

In [8]:
def tri_bisect(nums):
    """Cách 2: sort + tìm nhị phân vị trí đầu tiên có giá trị >= nums[i] + nums[j]."""
    nums = sorted(nums)
    n = len(nums)
    ans = 0
    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            left = bisect_left(nums, nums[i] + nums[j], j + 1)
            ans += left - j - 1
    return ans


for nums, expected in examples:
    print(nums, "->", tri_bisect(nums))
    assert tri_bisect(nums) == expected

[2, 2, 3, 4] -> 3
[4, 2, 3, 4] -> 4


**Code nộp LeetCode cho Cách 2:**

```python
from bisect import bisect_left

class Solution:
    def triangleNumber(self, nums: List[int]) -> int:
        nums.sort()
        n = len(nums)
        ans = 0
        for i in range(n - 2):
            for j in range(i + 1, n - 1):
                left = bisect_left(nums, nums[i] + nums[j], j + 1)
                ans += left - j - 1
        return ans
```

**Độ phức tạp:** thời gian `O(n² log n)` (`n²/2` cặp, mỗi cặp một lần tìm nhị phân); bộ nhớ `O(log n)` do sắp xếp (hoặc `O(n)` nếu `sorted`). Với `n = 1000`: `5·10^5 · 10 = 5·10^6` — nhanh.

---

### Cách 3 — Sắp xếp + hai con trỏ (cố định cạnh lớn nhất)

Giải thích từng đoạn:

1. `nums.sort()`.
2. Với `k` từ `n-1` xuống `2` (cạnh lớn nhất `nums[k]`): đặt `i = 0`, `j = k - 1`.
3. Khi `i < j`:
   - Nếu `nums[i] + nums[j] > nums[k]`: mọi chỉ số `i, i+1, ..., j-1` ghép với `j` đều hợp lệ ⇒ `ans += j - i`; rồi `j -= 1` (đã đếm hết các cặp có `j` làm cạnh vừa).
   - Ngược lại `i += 1` (cạnh nhỏ nhất chưa đủ lớn; mọi cặp `(i, j')` với `j' ≤ j` cũng không đủ vì `nums[j'] ≤ nums[j]`, nên bỏ `i` đi là an toàn).
4. Cộng dồn `ans`.

In [9]:
def tri_two_pointers(nums):
    """Cách 3: sort + hai con trỏ, cố định cạnh lớn nhất nums[k]."""
    nums = sorted(nums)
    n = len(nums)
    ans = 0
    for k in range(n - 1, 1, -1):
        i, j = 0, k - 1
        while i < j:
            if nums[i] + nums[j] > nums[k]:
                ans += j - i                  # mọi i' trong [i, j-1] ghép với j đều hợp lệ
                j -= 1
            else:
                i += 1
    return ans


for nums, expected in examples:
    print(nums, "->", tri_two_pointers(nums))
    assert tri_two_pointers(nums) == expected

[2, 2, 3, 4] -> 3
[4, 2, 3, 4] -> 4


**Code nộp LeetCode cho Cách 3:**

```python
class Solution:
    def triangleNumber(self, nums: List[int]) -> int:
        nums.sort()
        n = len(nums)
        ans = 0
        for k in range(n - 1, 1, -1):
            i, j = 0, k - 1
            while i < j:
                if nums[i] + nums[j] > nums[k]:
                    ans += j - i
                    j -= 1
                else:
                    i += 1
        return ans
```

**Độ phức tạp:** mỗi `k`, `i` chỉ tăng và `j` chỉ giảm nên vòng `while` chạy tối đa `k` bước ⇒ tổng **`O(n²)`**, bộ nhớ `O(1)` phụ (ngoài sắp xếp). Đây là cách nhanh nhất về bậc (`n²`).

---

### Cách 4 — Sắp xếp + bảng đếm trước (prefix count)

Ý tưởng: ở Cách 2, ta chỉ dùng `bisect_left` để biết "có bao nhiêu phần tử `< s`". Vì giá trị chỉ trong `[0, 1000]`, ta dựng sẵn bảng `less[s]` = **số phần tử `< s`** cho mọi `s ∈ [0, 2·max]` (tổng hai cạnh tối đa `2000`). Khi đó chỉ số `left` (đầu tiên `≥ s`) chính là `less[s]`, tra bảng `O(1)`.

Giải thích từng đoạn:

1. `mx = max(nums)`; `freq[v]` = số lần xuất hiện của `v`.
2. `less[s]` với `s = 0 .. 2·mx + 1`: `less[0] = 0`, `less[s+1] = less[s] + freq[s]` (chỉ cộng nếu `s ≤ mx`).
3. Sắp xếp `nums`; với mỗi cặp `i < j`: `s = nums[i] + nums[j]`; các `k > j` hợp lệ là `[j+1, less[s]-1]` ⇒ `ans += max(0, less[s] - (j + 1))`.

(`max(0, ...)` vì có thể `less[s] ≤ j`, ví dụ khi `nums[i] = 0`.)

In [10]:
def tri_prefix(nums):
    """Cách 4: sort + bảng less[s] = số phần tử < s (thay cho tìm nhị phân)."""
    nums = sorted(nums)
    n = len(nums)
    mx = nums[-1]
    freq = [0] * (mx + 1)
    for v in nums:
        freq[v] += 1
    less = [0] * (2 * mx + 2)                 # less[s] = số phần tử có giá trị < s
    for s in range(2 * mx + 1):
        less[s + 1] = less[s] + (freq[s] if s <= mx else 0)
    ans = 0
    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            ans += max(0, less[nums[i] + nums[j]] - (j + 1))
    return ans


for nums, expected in examples:
    print(nums, "->", tri_prefix(nums))
    assert tri_prefix(nums) == expected
print(tri_prefix([0, 0, 0]), tri_prefix([0, 1, 1]), tri_prefix([1, 1, 1]))   # 0 0 1

[2, 2, 3, 4] -> 3
[4, 2, 3, 4] -> 4
0 0 1


**Code nộp LeetCode cho Cách 4:**

```python
class Solution:
    def triangleNumber(self, nums: List[int]) -> int:
        nums.sort()
        n = len(nums)
        mx = nums[-1]
        freq = [0] * (mx + 1)
        for v in nums:
            freq[v] += 1
        less = [0] * (2 * mx + 2)
        for s in range(2 * mx + 1):
            less[s + 1] = less[s] + (freq[s] if s <= mx else 0)
        ans = 0
        for i in range(n - 2):
            for j in range(i + 1, n - 1):
                ans += max(0, less[nums[i] + nums[j]] - (j + 1))
        return ans
```

**Độ phức tạp:** dựng bảng `O(V)` với `V = max(nums) ≤ 1000`; vòng đếm `O(n²)`; bộ nhớ `O(V)`. Chỉ khả thi vì giá trị bị chặn nhỏ.

## 8. TRACE từng bước

### 8.1. Trace Cách 3 (hai con trỏ) với `nums = [2, 2, 3, 4]`

Mảng đã sắp xếp: `[2, 2, 3, 4]` (chỉ số 0..3).

| k | `nums[k]` | i | j | `nums[i]+nums[j]` | so với `nums[k]` | hành động | ans |
|---|---|---|---|---|---|---|---|
| 3 | 4 | 0 | 2 | 2+3=5 | > 4 | `ans += j-i = 2` (cặp (0,2),(1,2)); `j=1` | 2 |
| 3 | 4 | 0 | 1 | 2+2=4 | không > 4 | `i = 1` | 2 |
| 3 | 4 | 1 | 1 | — | `i<j` sai | dừng | 2 |
| 2 | 3 | 0 | 1 | 2+2=4 | > 3 | `ans += 1` (cặp (0,1)); `j=0` | 3 |
| 2 | 3 | 0 | 0 | — | dừng | | 3 |

Tổng `3` ✓. Hàm `trace_two_pointers` in từng dòng trên.

In [11]:
def trace_two_pointers(nums):
    nums = sorted(nums)
    n = len(nums)
    print("đã sort:", nums)
    ans = 0
    for k in range(n - 1, 1, -1):
        print(f"--- k={k} (cạnh lớn nhất = {nums[k]})")
        i, j = 0, k - 1
        while i < j:
            s = nums[i] + nums[j]
            if s > nums[k]:
                ans += j - i
                print(f"  i={i} j={j}: {nums[i]}+{nums[j]}={s} > {nums[k]} -> cộng {j - i} cặp, j={j - 1}, ans={ans}")
                j -= 1
            else:
                print(f"  i={i} j={j}: {nums[i]}+{nums[j]}={s} <= {nums[k]} -> i={i + 1}")
                i += 1
    return ans


assert trace_two_pointers([2, 2, 3, 4]) == 3
print("=======")
assert trace_two_pointers([4, 2, 3, 4]) == 4

đã sort: [2, 2, 3, 4]
--- k=3 (cạnh lớn nhất = 4)
  i=0 j=2: 2+3=5 > 4 -> cộng 2 cặp, j=1, ans=2
  i=0 j=1: 2+2=4 <= 4 -> i=1
--- k=2 (cạnh lớn nhất = 3)
  i=0 j=1: 2+2=4 > 3 -> cộng 1 cặp, j=0, ans=3
đã sort: [2, 3, 4, 4]
--- k=3 (cạnh lớn nhất = 4)
  i=0 j=2: 2+4=6 > 4 -> cộng 2 cặp, j=1, ans=2
  i=0 j=1: 2+3=5 > 4 -> cộng 1 cặp, j=0, ans=3
--- k=2 (cạnh lớn nhất = 4)
  i=0 j=1: 2+3=5 > 4 -> cộng 1 cặp, j=0, ans=4


### 8.2. Trace Cách 2 (tìm nhị phân) với `nums = [2, 3, 4, 4]` (ví dụ 2 đã sắp xếp)

| i | j | `s = nums[i]+nums[j]` | `bisect_left(nums, s, j+1)` | cộng `left-j-1` | ans |
|---|---|---|---|---|---|
| 0 | 1 | 5 | 4 (mọi phần tử từ chỉ số 2 đều `< 5`) | 4-1-1 = 2 | 2 |
| 0 | 2 | 6 | 4 | 4-2-1 = 1 | 3 |
| 1 | 2 | 7 | 4 | 4-2-1 = 1 | 4 |

(`j` chạy tới `n-2 = 2`.) Tổng `4` ✓.

In [12]:
def trace_bisect(nums):
    nums = sorted(nums)
    n = len(nums)
    print("đã sort:", nums)
    ans = 0
    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            s = nums[i] + nums[j]
            left = bisect_left(nums, s, j + 1)
            ans += left - j - 1
            print(f"i={i} j={j}: s={s}, left={left}, các k hợp lệ = {list(range(j + 1, left))}, ans={ans}")
    return ans


assert trace_bisect([4, 2, 3, 4]) == 4

đã sort: [2, 3, 4, 4]
i=0 j=1: s=5, left=4, các k hợp lệ = [2, 3], ans=2
i=0 j=2: s=6, left=4, các k hợp lệ = [3], ans=3
i=1 j=2: s=7, left=4, các k hợp lệ = [3], ans=4


## 9. Code nộp LeetCode

Chọn Cách 3 (hai con trỏ, `O(n²)`): đúng `class Solution`, phương thức `triangleNumber(self, nums: List[int]) -> int`.

In [13]:
class Solution:
    def triangleNumber(self, nums: List[int]) -> int:
        nums.sort()
        n = len(nums)
        ans = 0
        for k in range(n - 1, 1, -1):
            i, j = 0, k - 1
            while i < j:
                if nums[i] + nums[j] > nums[k]:
                    ans += j - i
                    j -= 1
                else:
                    i += 1
        return ans


sol = Solution()
print(sol.triangleNumber([2, 2, 3, 4]))     # 3
print(sol.triangleNumber([4, 2, 3, 4]))     # 4

3
4


## 10. Vì sao đúng

1. **Rút gọn điều kiện:** với `a ≤ b ≤ c` (mục 3.2), tam giác hợp lệ ⇔ `a + b > c`. Sắp xếp biến mọi bộ ba chỉ số `i < j < k` thành đúng một bộ `(nums[i], nums[j], nums[k])` đã có thứ tự `≤`; mỗi bộ ba **tập chỉ số** ứng với đúng một bộ `i < j < k` sau sắp xếp ⇒ không đếm trùng, không sót.
2. **Hai con trỏ đúng (cố định `k`):** bất biến: *mọi cặp `(i', j')` với `i' < j'`, `j' > j` đã được xử lý (đã đếm hoặc chắc chắn không hợp lệ), và mọi cặp có `i' < i` mà `j' ≤ j` chắc chắn không hợp lệ*.
   - Khi `nums[i] + nums[j] > nums[k]`: với mọi `i' ∈ [i, j-1]` ta có `nums[i'] ≥ nums[i]` nên `nums[i'] + nums[j] ≥ nums[i] + nums[j] > nums[k]` ⇒ cả `j - i` cặp `(i', j)` hợp lệ. Sau đó mọi cặp có `j` làm cạnh vừa đã xong ⇒ `j -= 1`.
   - Khi `nums[i] + nums[j] ≤ nums[k]`: với mọi `j' ≤ j` ta có `nums[i] + nums[j'] ≤ nums[i] + nums[j] ≤ nums[k]` ⇒ `i` không ghép được với bất kỳ `j'` nào còn lại ⇒ `i += 1` an toàn.
   - Mỗi cặp hợp lệ được đếm **đúng một lần** (ở bước mà `j` là cạnh vừa của nó).
3. **Cách 2 đúng:** với `(i, j)` cố định, tập `k > j` thoả `nums[k] < s` là một đoạn đầu của `[j+1, n-1]` vì mảng tăng; `bisect_left` cho đúng điểm kết thúc (`left` = đầu tiên `≥ s`, mọi chỉ số trước đó `< s`).
4. **Cách 4 đúng:** `less[s]` = số phần tử `< s` = chỉ số `left` của `bisect_left(nums, s)` khi tìm trên toàn mảng; vì các phần tử `k ≤ j` có giá trị `≤ nums[j] ≤ s` nên chỉ cần lấy `max(0, less[s] - (j+1))`.
5. **Trường hợp `0`:** nếu `nums[i] = 0` thì `s = nums[j]`; mọi `k > j` có `nums[k] ≥ nums[j] = s` ⇒ không hợp lệ (hai con trỏ: `0 + nums[j] > nums[k]` sai). Khớp với việc cạnh 0 không tạo tam giác.

## 11. Phân tích Time / Space Complexity

`n = len(nums)`, `V = max(nums) ≤ 1000`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Ba vòng lặp | `O(n³)` | `O(1)` | oracle, chậm với `n = 1000` |
| 2. Sort + bisect | `O(n² log n)` | `O(log n)`/`O(n)` | đơn giản, nhanh đủ |
| 3. Sort + hai con trỏ | `O(n²)` | `O(log n)`/`O(1)` | **tối ưu về bậc**, nộp bài |
| 4. Sort + bảng đếm | `O(n² + V)` | `O(V)` | dựa vào giá trị nhỏ |

Với `n = 1000`: Cách 1 ≈ `1.66·10^8` bước; Cách 2 ≈ `5·10^6` phép tìm nhị phân; Cách 3 ≈ `5·10^5` bước dịch con trỏ (mỗi `k` tối đa `k` bước, tổng `≈ n²/2`).

## 12. Bộ test

Chiến lược:

1. **Ví dụ đề** (cả 4 cách + `Solution`).
2. **Edge case:** `n < 3`, toàn số `0`, có `0` lẫn số khác, bộ suy biến `(1,2,3)`, mọi số bằng nhau, `n = 1000`.
3. **Phản ví dụ** cho các cách làm sai (xem mục 13).
4. **Đối chiếu nhiều input** bằng `assert`: oracle (Cách 1) với Cách 2, 3, 4 và `Solution` trên hàng nghìn input nhỏ; Cách 2, 3, 4 và `Solution` với nhau trên input đúng ràng buộc (`n` tới 1000, giá trị `0..1000`).
5. **Kiểm tra bằng công thức:** mọi số bằng nhau `v > 0` ⇒ `C(n, 3)`; toàn `0` ⇒ `0`.
6. **Hiệu năng:** `n = 1000`.

In [14]:
IMPLS = [tri_bruteforce, tri_bisect, tri_two_pointers, tri_prefix, lambda arr: Solution().triangleNumber(list(arr))]
NAMES = ["bruteforce", "bisect", "two_pointers", "prefix", "Solution"]

cases = [
    ([2, 2, 3, 4], 3),
    ([4, 2, 3, 4], 4),
    ([1], 0),
    ([1, 2], 0),
    ([0], 0),
    ([0, 0, 0], 0),                 # toàn 0
    ([0, 1, 1], 0),                 # cạnh 0 không tạo tam giác
    ([1, 1, 1], 1),
    ([1, 2, 3], 0),                 # suy biến 1+2=3
    ([2, 3, 4], 1),
    ([3, 4, 5], 1),
    ([1, 1, 1, 1], 4),              # C(4,3)
    ([2, 2, 3, 4, 5], 6),           # đã tính tay ở mục 18
    ([1, 1, 2, 2], 2),              # (1,2,2) dùng mỗi số 1
    ([1000, 1000, 1000], 1),
    ([1, 1000, 1000], 1),
    ([1, 2, 1000], 0),
]
for nums, expected in cases:
    for f, name in zip(IMPLS, NAMES):
        got = f(list(nums))
        assert got == expected, (name, nums, got, expected)
print(len(cases), "ca cố định: cả 5 cách đều đúng")

# Công thức: toàn số bằng nhau -> C(n, 3); toàn 0 -> 0
for n in (3, 4, 10, 50):
    for v in (1, 7, 1000):
        arr = [v] * n
        for f in IMPLS[1:]:
            assert f(list(arr)) == comb(n, 3)
    for f in IMPLS[1:]:
        assert f([0] * n) == 0
print("công thức C(n,3) cho mảng bằng nhau: OK")

17 ca cố định: cả 5 cách đều đúng
công thức C(n,3) cho mảng bằng nhau: OK


In [15]:
# Đối chiếu ngẫu nhiên, input NHỎ (có seed): oracle Cách 1 vs mọi cách khác
rng = random.Random(611)
for _ in range(3000):
    n = rng.randint(1, 12)
    hi = rng.choice([1, 2, 3, 5, 10, 1000])
    nums = [rng.randint(0, hi) for _ in range(n)]
    expected = tri_bruteforce(nums)
    for f, name in zip(IMPLS[1:], NAMES[1:]):
        assert f(list(nums)) == expected, (name, nums)
print("3000 input nhỏ ngẫu nhiên: mọi cách khớp oracle")

# So sánh với vét cạn kiểu khác: combinations + is_triangle (độc lập hơn nữa)
for _ in range(300):
    nums = [rng.randint(0, 15) for _ in range(rng.randint(1, 14))]
    expected = sum(1 for t in combinations(nums, 3) if is_triangle(*t))
    assert tri_two_pointers(nums) == expected
print("đối chiếu với combinations + is_triangle: OK")

# Đúng ràng buộc lớn: n tới 1000, giá trị 0..1000 — Cách 2, 3, 4, Solution khớp nhau
for _ in range(40):
    n = rng.randint(500, 1000)
    hi = rng.choice([10, 100, 1000])
    nums = [rng.randint(0, hi) for _ in range(n)]
    base = tri_two_pointers(nums)
    for f, name in zip([tri_bisect, tri_prefix, IMPLS[4]], ["bisect", "prefix", "Solution"]):
        assert f(list(nums)) == base, name
print("40 input lớn theo đúng ràng buộc: các cách khớp nhau")

3000 input nhỏ ngẫu nhiên: mọi cách khớp oracle


đối chiếu với combinations + is_triangle: OK


40 input lớn theo đúng ràng buộc: các cách khớp nhau


In [16]:
# Phân loại mọi bộ ba: hợp lệ + suy biến + bất khả = C(n,3) (kiểm tra tính nhất quán của việc đếm theo chỉ số)
rng = random.Random(7)
for _ in range(200):
    nums = [rng.randint(0, 8) for _ in range(rng.randint(3, 12))]
    valid = degenerate = impossible = 0
    for a, b, c in (sorted(t) for t in combinations(nums, 3)):
        if a + b > c:
            valid += 1
        elif a + b == c:
            degenerate += 1
        else:
            impossible += 1
    assert valid + degenerate + impossible == comb(len(nums), 3)
    assert valid == tri_two_pointers(nums)
print("hợp lệ + suy biến + bất khả = C(n,3): OK")

# Hiệu năng n = 1000
rng = random.Random(1)
worst = {
    "ngẫu nhiên 0..1000": [rng.randint(0, 1000) for _ in range(1000)],
    "toàn 1000": [1000] * 1000,
    "1..1000 tăng dần": list(range(1, 1001)),
}
for label, arr in worst.items():
    row = []
    for f, name in zip(IMPLS[1:], NAMES[1:]):
        t0 = time.perf_counter()
        r = f(list(arr))
        row.append((name, r, time.perf_counter() - t0))
    assert len({r for _, r, _ in row}) == 1
    print(f"{label:<20} đáp án {row[0][1]:>10}  " + "  ".join(f"{name}: {t:.3f}s" for name, _, t in row))

t0 = time.perf_counter()
tri_bruteforce(worst["ngẫu nhiên 0..1000"][:200])
print(f"(tham khảo) Cách 1 chỉ với n = 200 đã tốn {time.perf_counter() - t0:.3f}s")

hợp lệ + suy biến + bất khả = C(n,3): OK


ngẫu nhiên 0..1000   đáp án   83066700  bisect: 0.095s  two_pointers: 0.060s  prefix: 0.067s  Solution: 0.046s


toàn 1000            đáp án  166167000  bisect: 0.135s  two_pointers: 0.049s  prefix: 0.084s  Solution: 0.058s


1..1000 tăng dần     đáp án   82958750  bisect: 0.107s  two_pointers: 0.042s  prefix: 0.053s  Solution: 0.039s
(tham khảo) Cách 1 chỉ với n = 200 đã tốn 0.129s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `>=` thay vì `>` (đếm cả tam giác suy biến)

Điều kiện đúng là `a + b > c`. Với `>=`, bộ `(1, 2, 3)` (tổng `1 + 2 = 3`) bị tính là tam giác.

In [17]:
def wrong_ge(nums):
    nums = sorted(nums)
    n = len(nums)
    ans = 0
    for k in range(n - 1, 1, -1):
        i, j = 0, k - 1
        while i < j:
            if nums[i] + nums[j] >= nums[k]:        # SAI: phải là >
                ans += j - i
                j -= 1
            else:
                i += 1
    return ans


print(wrong_ge([1, 2, 3]), "(đúng: 0)")
print(wrong_ge([2, 2, 3, 4]), "(đúng: 3)")
print(wrong_ge([0, 0, 0]), "(đúng: 0)")

1 (đúng: 0)
4 (đúng: 3)
1 (đúng: 0)


**Sửa:** `nums[i] + nums[j] > nums[k]`. Luôn test các ca có **bằng** (suy biến) và **số 0**.

### Lỗi 2: quên sắp xếp, chỉ kiểm một bất đẳng thức theo thứ tự nhập

Chỉ kiểm tra `nums[i] + nums[j] > nums[k]` trên mảng **chưa sắp xếp** là kiểm tra thiếu: nếu `nums[k]` không phải cạnh lớn nhất thì hai điều kiện còn lại **không** tự đúng.

In [18]:
def wrong_no_sort(nums):
    n = len(nums)
    cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nums[i] + nums[j] > nums[k]:       # SAI: chưa sort nên chưa chắc nums[k] là cạnh lớn nhất
                    cnt += 1
    return cnt


print(wrong_no_sort([4, 2, 3, 4]), "(đúng: 4 — khớp do tình cờ, đừng tin vào ví dụ này)")
print(wrong_no_sort([10, 1, 1]), "(đúng: 0, vì 1 + 1 < 10)")
print(wrong_no_sort([1, 1, 10]), "(đúng: 0)")

4 (đúng: 4 — khớp do tình cờ, đừng tin vào ví dụ này)
1 (đúng: 0, vì 1 + 1 < 10)
0 (đúng: 0)


**Vì sao sai:** với `[10, 1, 1]`, kiểm `10 + 1 > 1` đúng nhưng `1 + 1 > 10` sai — bộ này không phải tam giác. **Sửa:** sắp xếp để `nums[k]` luôn là cạnh lớn nhất trong bộ, hoặc kiểm đủ ba điều kiện.

### Lỗi 3: cộng `1` thay vì `j - i` khi hai con trỏ gặp cặp hợp lệ

Khi `nums[i] + nums[j] > nums[k]`, **mọi** `i' ∈ [i, j-1]` đều hợp lệ với `j`. Cộng `1` bỏ sót `j - i - 1` cặp.

In [19]:
def wrong_plus_one(nums):
    nums = sorted(nums)
    n = len(nums)
    ans = 0
    for k in range(n - 1, 1, -1):
        i, j = 0, k - 1
        while i < j:
            if nums[i] + nums[j] > nums[k]:
                ans += 1                     # SAI: phải cộng j - i
                j -= 1
            else:
                i += 1
    return ans


print(wrong_plus_one([2, 2, 3, 4]), "(đúng: 3)")
print(wrong_plus_one([2, 3, 4, 4, 5, 6]), "(đúng:", tri_bruteforce([2, 3, 4, 4, 5, 6]), ")")

2 (đúng: 3)
8 (đúng: 16 )


**Sửa:** `ans += j - i` (cả đoạn `[i, j-1]`), sau đó `j -= 1`. Nếu chỉ cộng 1 thì phải **không** dịch `j` mà dịch `i`, như vậy sẽ chậm lại `O(n³)`.

### Lỗi 4: loại trùng bằng `set` (đề đếm theo **chỉ số**)

Ví dụ 1 nói rõ `(2,3,4)` xuất hiện **hai lần** vì lấy từ hai số `2` khác nhau. Nếu dùng `set(nums)` sẽ mất một số `2` và đếm sai.

In [20]:
def wrong_set(nums):
    return tri_two_pointers(list(set(nums)))      # SAI: bỏ trùng làm mất các bộ khác chỉ số


print(wrong_set([2, 2, 3, 4]), "(đúng: 3)")
print(wrong_set([4, 2, 3, 4]), "(đúng: 4)")
print(wrong_set([5, 5, 5, 5]), "(đúng: 4)")

1 (đúng: 3)
1 (đúng: 4)
0 (đúng: 4)


### Lỗi 5: đếm mỗi bộ ba nhiều lần (không ép `i < j < k`)

Nếu ba vòng lặp đều chạy `range(n)` và chỉ loại `i == j == k` bằng cách ép đôi một khác nhau, mỗi tập 3 chỉ số bị đếm `3! = 6` lần (6 hoán vị).

In [21]:
def wrong_permutations(nums):
    n = len(nums)
    cnt = 0
    for i in range(n):
        for j in range(n):
            for k in range(n):
                if len({i, j, k}) == 3 and is_triangle(nums[i], nums[j], nums[k]):   # SAI: thiếu i < j < k
                    cnt += 1
    return cnt


print(wrong_permutations([2, 2, 3, 4]), "= 6 x", tri_bruteforce([2, 2, 3, 4]))

18 = 6 x 3


**Sửa:** `j` chạy từ `i + 1`, `k` từ `j + 1` (mỗi tập chỉ số đúng một lần), hoặc chia kết quả cho 6 (không nên, dễ nhầm).

**Checklist chống lỗi:**

- [ ] Dùng `>` (không phải `>=`).
- [ ] Sắp xếp trước, rồi chỉ cần `a + b > c`.
- [ ] Hai con trỏ: cộng `j - i`, rồi `j -= 1`; ngược lại `i += 1`.
- [ ] Không loại trùng; đếm theo **chỉ số**.
- [ ] Ép `i < j < k`.

## 14. Mẹo & nhận dạng pattern

- **"Đếm bộ ba thoả điều kiện về tổng/bất đẳng thức"** ⇒ **sắp xếp**, cố định một phần tử (thường là phần tử lớn nhất hoặc nhỏ nhất), dùng **hai con trỏ** cho hai phần tử còn lại: `O(n³) → O(n²)`. Cùng họ: 3Sum (15), 3Sum Closest (16), 3Sum Smaller (259).
- **Tính đơn điệu sau khi sắp xếp** cho phép đếm **cả đoạn** (`j - i` cặp) trong một bước.
- **`bisect_left(a, x, lo)`** thay cho vòng tìm tuyến tính khi tìm "chỉ số đầu tiên `≥ x`".
- **Giá trị bị chặn nhỏ (`≤ 1000`)** ⇒ nghĩ tới **bảng đếm/prefix** thay cho tìm nhị phân.
- **Tam giác:** sau khi sắp xếp chỉ cần `a + b > c`; **suy biến** (`=`) không tính.
- **Đếm theo chỉ số:** khi đề cho phép giá trị trùng, đừng loại trùng nếu đề tính các chỉ số khác nhau là khác nhau.
- Bài liên quan: 976 (Largest Perimeter Triangle), 259, 15, 16, 18, 1 (Two Sum).

## 15. Biến thể

1. **Bài 976 — Largest Perimeter Triangle:** tìm chu vi lớn nhất của một tam giác lập được từ mảng. Sắp xếp **giảm dần**; tham lam: bộ ba liên tiếp đầu tiên thoả `a < b + c` cho chu vi lớn nhất.
2. **Đếm tam giác suy biến** (`a + b = c`): đổi `>` thành `==` (đếm khéo hơn bằng bảng tần suất).
3. **Đếm tam giác nhọn/vuông/tù:** thay điều kiện bằng `a² + b² > / = / < c²` (sau khi đã là tam giác).
4. **Giới hạn chu vi:** đếm bộ ba tạo tam giác có chu vi `≤ P` (thêm điều kiện, vẫn hai con trỏ nhưng cần cẩn thận).
5. **Bảng tần suất theo giá trị (`V ≤ 1000`):** đếm theo bộ giá trị `(a ≤ b ≤ c)` rồi nhân hệ số tổ hợp — `O(V²)`, độc lập với `n`.

Minh hoạ nhanh biến thể 1 và 2:

In [22]:
def largest_perimeter(nums):
    """Biến thể 1 (bài 976): chu vi lớn nhất hoặc 0."""
    a = sorted(nums, reverse=True)
    for i in range(len(a) - 2):
        if a[i] < a[i + 1] + a[i + 2]:
            return a[i] + a[i + 1] + a[i + 2]
    return 0


def count_degenerate(nums):
    """Biến thể 2: đếm bộ ba chỉ số có a + b == c (a <= b <= c)."""
    return sum(1 for t in combinations(nums, 3) if (lambda s: s[0] + s[1] == s[2])(sorted(t)))


print(largest_perimeter([2, 1, 2]), largest_perimeter([1, 2, 1, 10]), largest_perimeter([3, 6, 2, 3]))   # 5 0 8
print(count_degenerate([1, 2, 3]), count_degenerate([2, 2, 4, 4]), count_degenerate([1, 1, 2, 2]))      # 1 2 2
assert largest_perimeter([2, 1, 2]) == 5 and largest_perimeter([1, 2, 1, 10]) == 0 and largest_perimeter([3, 6, 2, 3]) == 8

5 0 8
1 2 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ba vòng lặp | duyệt mọi `i<j<k`, kiểm đủ ba bất đẳng thức | `O(n³)` | `O(1)` | chỉ để học / oracle |
| 2. Sort + bisect | cố định `(i, j)`, tìm nhị phân điểm cắt của `k` | `O(n² log n)` | `O(log n)` | dễ viết, đủ nhanh cho `n = 1000` |
| 3. Sort + hai con trỏ | cố định cạnh lớn nhất `k`, đếm cặp bằng đơn điệu | `O(n²)` | `O(1)` phụ | **mặc định khi nộp bài** |
| 4. Sort + bảng đếm | thay bisect bằng bảng `less[s]` | `O(n² + V)` | `O(V)` | giá trị nhỏ; muốn tránh `log` |

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có cell TODO để bạn tự làm (cell **chưa** được gọi, chỉ gán `None` hoặc `def` rỗng). Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Dự đoán (True/False) cho `is_triangle(3, 4, 5)`, `is_triangle(1, 2, 3)`, `is_triangle(5, 5, 9)`, `is_triangle(0, 7, 7)` và giá trị `Solution().triangleNumber([3, 3, 3])`. Ghi vào list `bai1_du_doan` theo đúng thứ tự.

In [23]:
# TODO Bài 1: bai1_du_doan = [..., ..., ..., ..., ...]
bai1_du_doan = None

### Bài 2 (tính tay)

`nums = [2, 2, 3, 4, 5]`: có bao nhiêu bộ ba chỉ số tạo tam giác? (Gợi ý: liệt kê theo giá trị, nhớ hai số `2` khác chỉ số.) Ghi vào `bai2`.

In [24]:
# TODO Bài 2: bai2 = ...
bai2 = None

### Bài 3 (điền TODO)

Hoàn thành hàm `count_pairs(nums, k)`: với `nums` **đã sắp xếp** và chỉ số `k`, trả số cặp `(i, j)`, `i < j < k`, thoả `nums[i] + nums[j] > nums[k]` bằng hai con trỏ.

In [25]:
def count_pairs(nums, k):
    # TODO: i, j = 0, k - 1 ; while i < j: nếu nums[i] + nums[j] > nums[k] thì cộng j - i và j -= 1, ngược lại i += 1
    pass

### Bài 4 (tự viết lại)

Không nhìn mục 9, tự viết hàm `triangle_number(nums)` (sort + hai con trỏ, cố định cạnh lớn nhất).

In [26]:
# TODO Bài 4: tự viết lại
def triangle_number(nums):
    pass

### Bài 5 (biến thể / mini challenge)

Viết hàm `largest_perimeter(nums)` (bài 976): trả **chu vi lớn nhất** của một tam giác lập được từ ba số trong `nums`, hoặc `0` nếu không có. Sau đó tự trả lời: vì sao chỉ cần xét các bộ ba **liền kề** sau khi sắp xếp giảm dần?

In [27]:
# TODO Bài 5
def largest_perimeter(nums):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: tam giác hợp lệ cần tổng hai cạnh nhỏ **lớn hơn hẳn** cạnh lớn nhất.
- Mức 2: `1+2=3` là suy biến; `0+7=7` cũng không lớn hơn 7.
- Mức 3: `[True, False, True, False, 1]`: với `[3,3,3]` chỉ có một bộ ba chỉ số `(0,1,2)`.

**Bài 2**
- Mức 1: có `C(5,3) = 10` bộ ba chỉ số; xét theo giá trị `(a, b, c)`.
- Mức 2: các bộ hợp lệ: `(2,2,3)`, `(2,3,4)` (hai cách chọn số 2), `(2,4,5)` (hai cách), `(3,4,5)`; loại `(2,2,4)`, `(2,2,5)`, `(2,3,5)` (hai cách).
- Mức 3: `1 + 2 + 2 + 1 = 6`.

**Bài 3**
- Mức 1: `i = 0`, `j = k - 1`, `cnt = 0`.
- Mức 2: trong `while i < j`: nếu tổng `> nums[k]` thì `cnt += j - i`, `j -= 1`.
- Mức 3: nhánh còn lại `i += 1`; cuối hàm `return cnt`.

**Bài 4**
- Mức 1: sắp xếp, vòng ngoài `for k in range(n - 1, 1, -1)`.
- Mức 2: dùng lại ý hàm `count_pairs` ở Bài 3 cho từng `k`.
- Mức 3: cộng kết quả của mọi `k` rồi trả.

**Bài 5**
- Mức 1: sắp xếp giảm dần.
- Mức 2: với mỗi `i`, kiểm tra `a[i] < a[i+1] + a[i+2]` (ba số liền kề).
- Mức 3: trả ngay chu vi bộ đầu tiên thoả (lớn nhất); nếu không có thì `0`. Lý do: nếu bộ `(a[i], a[i+1], a[i+2])` không hợp lệ (cạnh lớn nhất `a[i]` quá dài) thì `a[i]` không ghép được với bất kỳ hai số nhỏ hơn nào (tổng của hai số nhỏ hơn còn nhỏ hơn `a[i+1] + a[i+2]`); còn nếu hợp lệ thì thay `a[i+1]`/`a[i+2]` bằng số nhỏ hơn chỉ làm chu vi giảm.

## 19. Đáp án tham khảo

Chạy kiểm tra từng đáp án bằng `assert`.

In [28]:
# Bài 1
dap_an_1 = [is_triangle(3, 4, 5), is_triangle(1, 2, 3), is_triangle(5, 5, 9), is_triangle(0, 7, 7),
            Solution().triangleNumber([3, 3, 3])]
print(dap_an_1)
assert dap_an_1 == [True, False, True, False, 1]

# Bài 2
bai2_dap_an = tri_bruteforce([2, 2, 3, 4, 5])
print(bai2_dap_an)
assert bai2_dap_an == 6 and tri_two_pointers([2, 2, 3, 4, 5]) == 6

[True, False, True, False, 1]
6


In [29]:
# Bài 3
def count_pairs(nums, k):
    i, j = 0, k - 1
    cnt = 0
    while i < j:
        if nums[i] + nums[j] > nums[k]:
            cnt += j - i
            j -= 1
        else:
            i += 1
    return cnt


assert count_pairs([2, 2, 3, 4], 3) == 2 and count_pairs([2, 2, 3, 4], 2) == 1 and count_pairs([1, 2, 3], 2) == 0
print("Bài 3: OK")

# Bài 4
def triangle_number(nums):
    nums = sorted(nums)
    return sum(count_pairs(nums, k) for k in range(len(nums) - 1, 1, -1))


for nums, expected in cases:
    assert triangle_number(nums) == expected
rng = random.Random(4)
for _ in range(500):
    nums = [rng.randint(0, 20) for _ in range(rng.randint(1, 15))]
    assert triangle_number(nums) == tri_bruteforce(nums)
print("Bài 4: OK")

# Bài 5
def largest_perimeter(nums):
    a = sorted(nums, reverse=True)
    for i in range(len(a) - 2):
        if a[i] < a[i + 1] + a[i + 2]:
            return a[i] + a[i + 1] + a[i + 2]
    return 0


assert largest_perimeter([2, 1, 2]) == 5 and largest_perimeter([1, 2, 1, 10]) == 0
rng = random.Random(5)
for _ in range(1000):
    nums = [rng.randint(1, 30) for _ in range(rng.randint(1, 10))]
    best = max((sum(t) for t in combinations(nums, 3) if is_triangle(*t)), default=0)
    assert largest_perimeter(nums) == best
print("Bài 5: OK")

Bài 3: OK
Bài 4: OK
Bài 5: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "sắp xếp + cố định phần tử lớn nhất + hai con trỏ đếm cả đoạn":**

```python
class Solution:
    def triangleNumber(self, nums: List[int]) -> int:
        nums.sort()                               # a <= b <= c  =>  chỉ cần a + b > c
        n = len(nums)
        ans = 0
        for k in range(n - 1, 1, -1):             # nums[k] là cạnh lớn nhất
            i, j = 0, k - 1
            while i < j:
                if nums[i] + nums[j] > nums[k]:
                    ans += j - i                  # mọi i' trong [i, j-1] ghép với j đều được
                    j -= 1
                else:
                    i += 1                        # cạnh nhỏ nhất chưa đủ lớn
        return ans
```

**Checklist trước khi submit:**

- [ ] Đã **sắp xếp** trước và chỉ kiểm `a + b > c`?
- [ ] Dùng `>` chặt (loại tam giác suy biến, loại cạnh `0`)?
- [ ] Hai con trỏ: `ans += j - i` rồi `j -= 1`; ngược lại `i += 1`?
- [ ] Đếm theo **chỉ số** (không `set`, không loại trùng), mỗi tập chỉ số một lần?
- [ ] `k` chạy từ `n-1` xuống `2`; `n < 3` trả `0` (vòng lặp không chạy)?
- [ ] Đã test: `[0,0,0]`, `[1,2,3]`, mảng toàn số bằng nhau (`C(n,3)`), `n = 1000`, ví dụ đề?
- [ ] Đã đối chiếu với oracle ba vòng lặp trên input nhỏ ngẫu nhiên có seed?